In [0]:
dbutils.widgets.dropdown("environment", "dev", ["dev", "preprod", "prod"], "Environment")

In [0]:
def load_config_from_widget():
    try:
        environment = dbutils.widgets.get("environment")
    except:
        environment = "dev"
    
    storage_account = "databricksete123"
    project_name = "orders-analytics-pipeline"
    
    if environment == 'dev':
        config = {
            "environment": "dev",
            "catalog": "databricks_catalog_new",
            "schemas": {"bronze": "dev_bronze", "silver": "dev_silver", "gold": "dev_gold"},
            "storage": {
                "account": storage_account,
                "landing": {
                    "orders": f"abfss://source@{storage_account}.dfs.core.windows.net/orders/",
                    "customers": f"abfss://source@{storage_account}.dfs.core.windows.net/customers/",
                    "products": f"abfss://source@{storage_account}.dfs.core.windows.net/products/",
                    "region": f"abfss://source@{storage_account}.dfs.core.windows.net/region/"
                },
                "external": {
                    "bronze": f"abfss://claude-bronze@{storage_account}.dfs.core.windows.net/dev/{project_name}",
                    "silver": f"abfss://claude-silver@{storage_account}.dfs.core.windows.net/dev/{project_name}",
                    "gold": f"abfss://claude-gold@{storage_account}.dfs.core.windows.net/dev/{project_name}"
                }
            }
        }
    elif environment == 'preprod':
        config = {
            "environment": "preprod",
            "catalog": "databricks_catalog_new",
            "schemas": {"bronze": "preprod_bronze", "silver": "preprod_silver", "gold": "preprod_gold"},
            "storage": {
                "account": storage_account,
                "landing": {
                    "orders": f"abfss://source@{storage_account}.dfs.core.windows.net/orders/",
                    "customers": f"abfss://source@{storage_account}.dfs.core.windows.net/customers/",
                    "products": f"abfss://source@{storage_account}.dfs.core.windows.net/products/",
                    "region": f"abfss://source@{storage_account}.dfs.core.windows.net/region/"
                },
                "external": {
                    "bronze": f"abfss://claude-bronze@{storage_account}.dfs.core.windows.net/preprod/{project_name}",
                    "silver": f"abfss://claude-silver@{storage_account}.dfs.core.windows.net/preprod/{project_name}",
                    "gold": f"abfss://claude-gold@{storage_account}.dfs.core.windows.net/preprod/{project_name}"
                }
            }
        }
    else:
        config = {
            "environment": "prod",
            "catalog": "databricks_catalog_new",
            "schemas": {"bronze": "prod_bronze", "silver": "prod_silver", "gold": "prod_gold"},
            "storage": {
                "account": storage_account,
                "landing": {
                    "orders": f"abfss://source@{storage_account}.dfs.core.windows.net/orders/",
                    "customers": f"abfss://source@{storage_account}.dfs.core.windows.net/customers/",
                    "products": f"abfss://source@{storage_account}.dfs.core.windows.net/products/",
                    "region": f"abfss://source@{storage_account}.dfs.core.windows.net/region/"
                },
                "external": {
                    "bronze": f"abfss://claude-bronze@{storage_account}.dfs.core.windows.net/prod/{project_name}",
                    "silver": f"abfss://claude-silver@{storage_account}.dfs.core.windows.net/prod/{project_name}",
                    "gold": f"abfss://claude-gold@{storage_account}.dfs.core.windows.net/prod/{project_name}"
                }
            }
        }
    
    return config

config = load_config_from_widget()
print(f"🧹 Silver Orders Pipeline - {config['environment'].upper()} Environment")
print("Status: Cleaning and validating data...")

In [0]:
from pyspark.sql.functions import col, trim

bronze_table = f"{config['catalog']}.{config['schemas']['bronze']}.orders_raw"
print(f"📂 Reading from: {bronze_table}")

df_bronze = spark.table(bronze_table)

df_clean = df_bronze \
    .filter(col("total_amount") > 0) \
    .filter(col("quantity") > 0) \
    .withColumn("customer_id", trim(col("customer_id"))) \
    .withColumn("product_id", trim(col("product_id")))

silver_schema = config['schemas']['silver']
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {config['catalog']}.{silver_schema}")

external_path = f"{config['storage']['external']['silver']}/orders_clean"
silver_table = f"{config['catalog']}.{silver_schema}.orders_clean"

spark.sql(f"DROP TABLE IF EXISTS {silver_table}")

print(f"📂 Writing to: {external_path}")

df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .option("overwriteSchema", "true") \
    .option("path", external_path) \
    .saveAsTable(silver_table)

print(f"✅ {silver_table}: {spark.table(silver_table).count()} records")